In [ ]:
# Import core libraries for data handling, plotting, and modelling
import pandas as pd
import numpy as np

# Set matplotlib backend so plots can be saved without opening a GUI window
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# Import regression models
from sklearn.linear_model import LinearRegression, Ridge

# Import preprocessing and train-test split tools
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Import evaluation metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Import statistical testing
from scipy import stats

# Ignore warnings to keep notebook output clean
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load NT population dataset
pop = pd.read_excel('nt-government-regions_1986-to-2025.xlsx')

# Load crime datasets
crime_nov = pd.read_csv('nt_crime_statistics_nov_2023.csv')   # Historical file
crime_dec = pd.read_csv('nt_crime_statistics_dec_2025.csv')   # Newer file

# Convert column names in all datasets to snake_case for consistency
for df in [crime_nov, crime_dec, pop]:
    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]

In [ ]:
# Keep only comparable years from the November 2023 dataset
crime_nov_hist = (
    crime_nov[crime_nov['year'] <= 2022]
    .copy()
    .rename(columns={'reporting_region': 'region'})
)

# Remove unknown regions from the newer dataset and fix column names
crime_dec_new = (
    crime_dec[crime_dec['reporting_region'] != 'Unknown']
    .copy()
    .rename(columns={
        'reporting_region': 'region',
        'offence_type_': 'offence_type'
    })
)

In [ ]:
# Map local reporting regions to NT government population regions
region_map = {
    'Alice Springs': 'Central Australia',
    'Darwin': 'Greater Darwin',
    'Palmerston': 'Greater Darwin',
    'Katherine': 'Big Rivers',
    'Tennant Creek': 'Barkly',
    'Nhulunbuy': 'East Arnhem',
    'NT Balance': 'Top End'
}

# Create a population-region column in both crime datasets
for df in [crime_nov_hist, crime_dec_new]:
    df['pop_region'] = df['region'].map(region_map)

In [ ]:
# Function to aggregate total offences by year, month, region, and offence category
def aggregate_crime(df):
    return (
        df.groupby(['year', 'month_number', 'pop_region', 'region', 'offence_category'])
          .agg(total_offences=('number_of_offences', 'sum'))
          .reset_index()
    )

# Combine both cleaned crime datasets into one aggregated dataset
crime_all = (
    pd.concat([
        aggregate_crime(crime_nov_hist),
        aggregate_crime(crime_dec_new)
    ], ignore_index=True)
    .drop_duplicates()
)

# Aggregate again to get total monthly offences by region
crime_rym = (
    crime_all.groupby(['year', 'month_number', 'pop_region', 'region'])
             .agg(total_offences=('total_offences', 'sum'))
             .reset_index()
)

In [ ]:
# Function to count alcohol-related and domestic-violence-related offences
def get_involvement_flags(df):
    df2 = df.copy()

    # Convert Yes/No flags into 1/0 values
    df2['alc'] = (
        df2['alcohol_involvement']
        .str.lower()
        .str.contains('yes')
        .fillna(False)
        .astype(int)
    )

    df2['dv'] = (
        df2['dv_involvement']
        .str.lower()
        .str.contains('yes')
        .fillna(False)
        .astype(int)
    )

    # Aggregate counts by time and region
    return (
        df2.groupby(['year', 'month_number', 'pop_region', 'region'])
           .agg(
               alc_offences=('alc', 'sum'),
               dv_offences=('dv', 'sum')
           )
           .reset_index()
    )

# Combine involvement counts from both datasets
flags_all = (
    pd.concat([
        get_involvement_flags(crime_nov_hist),
        get_involvement_flags(crime_dec_new)
    ])
    .drop_duplicates()
)

# Merge flag counts into the regional monthly crime table
crime_rym = crime_rym.merge(
    flags_all,
    on=['year', 'month_number', 'pop_region', 'region'],
    how='left'
)

In [ ]:
# Aggregate total population by year and region
pop_total = (
    pop.groupby(['year', 'region'])['population']
       .sum()
       .reset_index()
       .rename(columns={'region': 'pop_region'})
)

# Merge crime data with population data
merged = (
    crime_rym.merge(pop_total, on=['year', 'pop_region'], how='left')
             .dropna(subset=['population'])
)

In [ ]:
# Create offence rate per 1,000 residents
merged['offence_rate_per_1000'] = (merged['total_offences'] / merged['population']) * 1000

# Create proportional alcohol and domestic violence involvement rates
merged['alc_rate'] = merged['alc_offences'] / merged['total_offences'].replace(0, np.nan)
merged['dv_rate'] = merged['dv_offences'] / merged['total_offences'].replace(0, np.nan)

# Log-transform population to reduce skewness
merged['log_population'] = np.log(merged['population'])

# Create a year index for temporal trend modelling
merged['year_index'] = merged['year'] - merged['year'].min()

# Replace remaining missing values with 0
merged = merged.fillna(0)

In [ ]:
# Create demographic summary with Aboriginal population percentage
pop_demo = (
    pop.groupby(['year', 'region'])
       .apply(lambda g: pd.Series({
           'total_pop': g['population'].sum(),
           'aboriginal_pop': g.loc[
               g['aboriginal_status'].str.contains('Aboriginal', na=False),
               'population'
           ].sum()
       }))
       .reset_index()
       .rename(columns={'region': 'pop_region'})
)

# Calculate Aboriginal population percentage
pop_demo['aboriginal_pct'] = (pop_demo['aboriginal_pop'] / pop_demo['total_pop']) * 100

# Merge demographic data into the main dataset
merged2 = (
    merged.merge(
        pop_demo[['year', 'pop_region', 'aboriginal_pct']],
        on=['year', 'pop_region'],
        how='left'
    )
    .fillna(0)
)

# Quick dataset summary
print(f"Final merged dataset: {merged.shape[0]} rows × {merged.shape[1]} cols")
print(f"Year range: {merged['year'].min()}–{merged['year'].max()}")
print(f"Regions: {sorted(merged['region'].unique())}")
print(f"Null values:\n{merged.isnull().sum()}")

In [ ]:
# How accurately can monthly offence counts be predicted
# using recent crime history and demographic features?

# Sort data by region and time
rq3 = merged.sort_values(['region', 'year', 'month_number']).copy()

# Create lag-based predictive features
rq3['prev_month_offences'] = rq3.groupby('region')['total_offences'].shift(1)
rq3['prev_3mo_avg'] = (
    rq3.groupby('region')['total_offences']
       .transform(lambda x: x.shift(1).rolling(3).mean())
)

# Drop rows with missing lag values
rq3 = rq3.dropna()

# Select predictor variables
features_rq3 = [
    'prev_month_offences',
    'prev_3mo_avg',
    'log_population',
    'alc_rate',
    'dv_rate',
    'month_number',
    'year_index'
]

X3 = rq3[features_rq3].values
y3 = rq3['total_offences'].values

# Split data using time order (no shuffling)
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.2, random_state=42, shuffle=False
)

# Standardise predictor values
scaler3 = StandardScaler()
X3s = scaler3.fit_transform(X3_train)
X3ts = scaler3.transform(X3_test)

# Train OLS model
lr3 = LinearRegression()
lr3.fit(X3s, y3_train)
y3_pred = lr3.predict(X3ts)

# Train Ridge model
ridge = Ridge(alpha=1.0)
ridge.fit(X3s, y3_train)
y3_pred_ridge = ridge.predict(X3ts)

# Evaluate OLS
mae3_lr = mean_absolute_error(y3_test, y3_pred)
rmse3_lr = np.sqrt(mean_squared_error(y3_test, y3_pred))
r2_3_lr = r2_score(y3_test, y3_pred)

# Evaluate Ridge
mae3_r = mean_absolute_error(y3_test, y3_pred_ridge)
rmse3_r = np.sqrt(mean_squared_error(y3_test, y3_pred_ridge))
r2_3_r = r2_score(y3_test, y3_pred_ridge)

# Compare absolute residual errors using Welch's t-test
res_lr = np.abs(y3_test - y3_pred)
res_ridge = np.abs(y3_test - y3_pred_ridge)
t_comp, p_comp = stats.ttest_ind(res_lr, res_ridge, equal_var=False)

# Print results
print("\n=== RQ3: Predictive Regression Results ===")
print(f"OLS:   MAE={mae3_lr:.1f}  RMSE={rmse3_lr:.1f}  R²={r2_3_lr:.3f}")
print(f"Ridge: MAE={mae3_r:.1f}  RMSE={rmse3_r:.1f}  R²={r2_3_r:.3f}")
print(f"Welch t-test OLS vs Ridge: t={t_comp:.4f}, p={p_comp:.4f}")
print(f"Interpretation: p={p_comp:.3f} > 0.05, so there is no significant difference in model error.")
print("OLS can be preferred because it is simpler and easier to interpret.")